In [13]:
import tensorflow as tf
import TensorSlider as ts
import keras


In [14]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [15]:
def createLabels(data, lookforward):
    """
    create input labels from the lookaehad data
    """

    ohlc = lookforward
    ohlc /= ohlc[0,:] # divide by open price
    ohlc -= 1 # zero out
    ohlc *= 100 # convert to 1/10 percentage, so 1 = 10 percent
    ohlc = tf.clip_by_value(ohlc,-1,1)
    high = ohlc[tf.math.argmax(ohlc[:,1]), 1]
    low = ohlc[tf.math.argmin(ohlc[:,2]), 2]


    # same process, compute relative deltas
    processed = []
    latestClose = data[0,3,0] # base timeframe
    pricemult = 10 # 1 = 10 percent
    volmult = 0.1
    for i in range(0, data.shape[0]):
        # lets go timeframe by timeframe
        prices = tf.divide(data[i,0:4,:], latestClose) - 1
        prices = tf.multiply(prices, pricemult)
        volumes = tf.divide(data[i,4,:], data[i,4,0]) - 1
        volumes = tf.multiply(volumes, volmult)
        volumes = tf.expand_dims(volumes, 0)
        processed.append(tf.concat([prices, volumes], axis=0))

    datares = tf.clip_by_value(tf.stack(processed, axis=0), clip_value_min=-1, clip_value_max=1)

    # remove nans
    datares = tf.keras.ops.nan_to_num(datares)

    return tf.expand_dims(datares,0), [low, high]

def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds

## Get Datasets

In [19]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 100
lookahead = 5

def getSlider(coin):

    path = tfrecordpath + coin+"/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return ts.WindowSlider(datasets, windowsize, lookahead)

def getSliders(coins):
    datasets = []
    for coin in coins:
        datasets.append(tf.data.Dataset.from_generator(lambda: getSlider(coin),
            output_signature=(
                tf.TensorSpec((5,5,windowsize), dtype=tf.float32),
                tf.TensorSpec((lookahead+1,5), dtype=tf.float32))
            ))#.prefetch(tf.data.AUTOTUNE))
    return datasets

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP"]
datasets = getSliders(coins)

## Combine Datasets

In [22]:
ds = tf.data.Dataset.sample_from_datasets(datasets = datasets).prefetch(tf.data.AUTOTUNE)
ds = datasets[0].prefetch(tf.data.AUTOTUNE)

In [25]:
import keras
import os

def load_model(name):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    # Check if we have any checkpoints in the first place
    if not os.path.exists(folder + "checkpoints"):
        # No checkpoint folder, lets create one, and return the model, epoch 0
        os.mkdir(folder + "checkpoints")
        return model, 0

    # Check how many epoch checkpoints we have in the (existing!) checkpoint folder
    checkpoints = os.listdir("models/" + name + "/checkpoints/")

    # We do this by just counting how many files are in there, we assume there will be no vandalism
    # and all files inside the checkpoint folder are checkpoints
    lastEpoch = len(checkpoints)

    # load the latest checkpoint if we have more than 1 of them
    if lastEpoch >= 1:
        model.load_weights("models/" + name + "/checkpoints/" + str(lastEpoch) + ".weights.h5")

    return model, lastEpoch

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, name, lastEpoch=0):
        super().__init__()
        # no idea if we want to or need to super this
        self.lastEpoch = lastEpoch
        self.name = name
        print("Model was trained for " + str(lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.model.save_weights("models/" + str(self.name) + "/checkpoints/" + str(self.lastEpoch) + ".weights.h5")
        print("\nSaved checkpoint of epoch number " + str(self.lastEpoch))
        # Save the latest model
        self.model.save("models/" + str(self.name) + "/model.keras")


In [28]:
modelName = "modeltest"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=0,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=False,
                                                update_freq="epoch",
                                                profile_batch=0,
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )

model, epochs = load_model(modelName)

history = model.fit(ds.map(createLabels), epochs=1, verbose=1, validation_data=None, callbacks=[saveEachEpoch(modelName, epochs), tensorboard])

Model was trained for 0 epochs before.
    381/Unknown 13s 17ms/step - MeanAbsolutePercentageError: 84762824.0000 - MeanSquaredError: 0.1254 - loss: 0.1254

KeyboardInterrupt: 